In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict

# CONFIG

GRID_SIZE = 8
MAX_STEPS = 100
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

# PIECES

def get_pieces(multiplier: int = 3) -> List[np.ndarray]:
    base = [
        np.array([[1, 1],
                  [1, 1]]),

        np.array([[1, 1, 1]]),

        np.array([[1, 0],
                  [1, 1]]),

        np.array([[0, 1],
                  [1, 1]]),

        np.array([[1, 1, 0],
                  [0, 1, 1]])
    ]
    return base * multiplier

# ENVIRONMENT

class FabricEnv:

    def __init__(self, pieces: List[np.ndarray]):
        self.pieces = pieces
        self._action_cache: Dict[Tuple, List] = {}

    def reset(self):
        self.grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=np.int8)
        self.remaining = random.sample(self.pieces, len(self.pieces))
        self._action_cache.clear()
        return self._get_state()

    def _get_state(self):
        return tuple(self.grid.flatten())

    def _get_state_key(self):
        grid_key = tuple(self.grid.flatten())
        pieces_key = tuple(tuple(p.flatten()) for p in self.remaining)
        return (grid_key, pieces_key)

    def _rotate(self, piece, r):
        return piece if r == 0 else np.rot90(piece)

    def _valid(self, piece, x, y):
        h, w = piece.shape
        if x + w > GRID_SIZE or y + h > GRID_SIZE:
            return False
        return not np.any(self.grid[y:y+h, x:x+w] + piece > 1)

    def _place(self, piece, x, y):
        h, w = piece.shape
        self.grid[y:y+h, x:x+w] += piece

    def _count_holes(self):
        holes = 0
        for y in range(1, GRID_SIZE - 1):
            for x in range(1, GRID_SIZE - 1):
                if self.grid[y, x] == 0:
                    if np.sum(self.grid[y-1:y+2, x-1:x+2]) >= 5:
                        holes += 1
        return holes

    def render(self):
        plt.imshow(self.grid, cmap="gray")
        plt.title("Final Layout")
        plt.axis("off")
        plt.show()

    def get_actions(self):
        state_key = self._get_state_key()

        if state_key in self._action_cache:
            return self._action_cache[state_key]

        actions = []

        for i, piece in enumerate(self.remaining):
            for r in (0, 1):
                rotated = self._rotate(piece, r)
                h, w = rotated.shape

                for y in range(GRID_SIZE - h + 1):
                    for x in range(GRID_SIZE - w + 1):
                        if self._valid(rotated, x, y):
                            actions.append((i, r, x, y))

        if not actions:
            actions = [None]

        self._action_cache[state_key] = actions
        return actions

    def step(self, action):
        if action is None:
            return self._get_state(), -10.0, True

        i, r, x, y = action
        piece = self._rotate(self.remaining[i], r)

        self._place(piece, x, y)
        self.remaining.pop(i)

        filled = np.sum(self.grid)
        holes = self._count_holes()

        # reward
        reward = 2 * filled - 3 * holes

        done = len(self.remaining) == 0

        if done:
            waste = np.sum(self.grid == 0)
            reward -= 2 * waste

        self._action_cache.clear()

        return self._get_state(), reward, done


# AGENT

@dataclass
class QParams:
    alpha: float = 0.1
    gamma: float = 0.95
    epsilon: float = 1.0
    epsilon_decay: float = 0.99
    epsilon_min: float = 0.05


class QLearningAgent:

    def __init__(self, params: QParams):
        self.params = params
        self.q: Dict[Tuple, float] = {}

    def _get_q(self, s, a):
        return self.q.get((s, a), 0.0)

    def act(self, env, state):
        actions = env.get_actions()

        if random.random() < self.params.epsilon:
            return random.choice(actions)

        values = [self._get_q(state, a) for a in actions]
        return actions[int(np.argmax(values))]

    def update(self, s, a, r, s2, env):
        actions2 = env.get_actions()
        max_q = max((self._get_q(s2, a2) for a2 in actions2), default=0.0)

        old = self._get_q(s, a)
        self.q[(s, a)] = old + self.params.alpha * (
            r + self.params.gamma * max_q - old
        )

    def decay(self):
        if self.params.epsilon > self.params.epsilon_min:
            self.params.epsilon *= self.params.epsilon_decay


# TRAINING

def moving_average(x, w=100):
    return np.convolve(x, np.ones(w)/w, mode='valid')


def train(env, agent, episodes=3000):
    rewards = []
    best_grid = None
    best_util = -1

    for ep in range(episodes):
        state = env.reset()
        total = 0

        for _ in range(MAX_STEPS):
            action = agent.act(env, state)
            next_state, reward, done = env.step(action)

            agent.update(state, action, reward, next_state, env)

            state = next_state
            total += reward

            if done:
                break

        agent.decay()
        rewards.append(total)

        util = np.mean(env.grid)
        if util > best_util:
            best_util = util
            best_grid = env.grid.copy()

        if ep % 200 == 0:
            print(f"Episode {ep} | reward={total:.2f} | avg={np.mean(rewards[-100:]):.2f}")

    return rewards, best_grid


# BASELINE

def random_policy(env):
    env.reset()
    for _ in range(MAX_STEPS):
        action = random.choice(env.get_actions())
        _, _, done = env.step(action)
        if done:
            break
    return np.mean(env.grid)


# MAIN

if __name__ == "__main__":
    pieces = get_pieces()

    env = FabricEnv(pieces)
    agent = QLearningAgent(QParams())

    rewards, best_grid = train(env, agent)

    # graphical results
    plt.plot(rewards, alpha=0.3, label="raw")
    plt.plot(moving_average(rewards), label="moving avg")
    plt.legend()
    plt.title("Training Reward")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.show()

    print("\n    BEST RESULT    ")
    print("Best utilization:", round(np.mean(best_grid), 3))

    plt.imshow(best_grid, cmap="gray")
    plt.title("Best Layout")
    plt.axis("off")
    plt.show()

    print("\n   RANDOM BASELINE  ")
    print("Random utilization:", round(random_policy(env), 3))